In [ ]:
#%pip install pandas
#%pip install scikit-learn

In [ ]:
!python --version

In [ ]:
import sklearn

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [ ]:
table=pd.read_csv('./exponents/exponents_2.csv')

In [ ]:
table.shape

In [ ]:
table

In [ ]:
def create_data(table,Data):
    new = pd.DataFrame(Data)
    return pd.concat([table,new],ignore_index=True)

def delete_data(table,idx=-1):
    if idx==-1:
        return table.drop(table.shape[0]-1,axis=0)
    else:
        return table.drop(idx,axis=0)
    

In [ ]:
import random
category={'지수':1,'로그':2}
data_cnt=random.randint(2000,3000) # 문제 수
INPUT_DATA={'delaytime':[],'category':[],'category_detail':[],'base':[],'exponents':[],'try':[]}
for i in range(data_cnt):
    init_delaytime=random.randint(3,60)
    base=random.randint(1,5)
    exponents=random.randint(1,5)
    INPUT_DATA['delaytime'].append(init_delaytime)
    INPUT_DATA['category'].append(category['지수'])
    INPUT_DATA['category_detail'].append(2) # 2. 지수 함수 값 맞추기
    INPUT_DATA['base'].append(base)
    INPUT_DATA['exponents'].append(exponents)
    INPUT_DATA['try'].append(random.randint(0,3))
        


train_table=create_data(table,INPUT_DATA)    
    
    
data_cnt=10 # 문제 수
OUTPUT_DATA={'delaytime':[],'category':[],'category_detail':[],'base':[],'exponents':[],'try':[]}
for i in range(data_cnt):
    init_delaytime=random.randint(3,60)
    base=random.randint(1,5)
    exponents=random.randint(1,5)
    OUTPUT_DATA['delaytime'].append(init_delaytime)
    OUTPUT_DATA['category'].append(category['지수'])
    OUTPUT_DATA['category_detail'].append(2) # 2. 지수 함수 값 맞추기
    OUTPUT_DATA['base'].append(base)
    OUTPUT_DATA['exponents'].append(exponents)
    OUTPUT_DATA['try'].append(random.randint(0,3))

val_table=create_data(table,OUTPUT_DATA)

In [ ]:
train_table

In [ ]:
val_table

### 정확도 = 1/시도횟수


https://velog.io/@supremo7/%EB%A8%B8%EC%8B%A0-%EB%9F%AC%EB%8B%9D-%EB%8B%A4%EC%A4%91-%EC%84%A0%ED%98%95-%ED%9A%8C%EA%B7%80-Multiple-Linear-Regression

In [ ]:
def table_divide(table):
    input_data={'delaytime':[],'base':[],'exponents':[]}
    output_data={'try':[]}

    for i in range(table.shape[0]):
        table_data=table.loc[i]
        input_data['delaytime'].append(table_data['delaytime'])
        input_data['base'].append(table_data['base'])
        input_data['exponents'].append(table_data['exponents'])
        
        output_data['try'].append(table_data['try'])
        
    X = pd.DataFrame(input_data)
    Y = pd.DataFrame(output_data)
    return X,Y

In [ ]:
X_train,y_train = table_divide(train_table)
X_test, y_test = table_divide(val_table)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
y_train.shape

In [ ]:
y_test.shape

In [ ]:
linear_regression_model = LinearRegression()  # 선형 회귀 모델을 가지고 오고 
linear_regression_model.fit(X_train, y_train)  # 학습 데이터를 이용해서 모델을 학습 시킨다

y_test_predict = linear_regression_model.predict(X_test)  # 학습시킨 모델로 예측

# 평균 제곱 오차의 루트를 통해서 테스트 데이터에서의 모델 성능 판단
mse = mean_squared_error(y_test, y_test_predict)

mse ** 0.5

In [ ]:
X_test

In [ ]:
y_test

In [ ]:
y_test_predict


In [ ]:
(1/y_test_predict)

In [ ]:
(1/y_test_predict)*100

## Pytorch 신경망 구현

In [ ]:
import random
import json
import numpy as np
import sys
import requests
from PIL import Image
import matplotlib.pyplot as plt
import torch.optim as optim
import shutil
import copy
import time
from torch.utils.data import Dataset
import xml.etree.ElementTree as et
from torchvision import transforms
from torch.utils.data import DataLoader


import torch
import torch.nn as nn

from torchvision.models import * 
from torch.optim import lr_scheduler

import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

![image.png](https://img1.daumcdn.net/thumb/R1280x0/?scode=mtistory2&fname=https%3A%2F%2Fblog.kakaocdn.net%2Fdn%2FbjQeTw%2FbtqFtiu9CNE%2FyAjNCKowd16OXIkbdZkkAK%2Fimg.png)

In [ ]:
class NvidiaModel(nn.Module):
    def __init__(self):
        super(NvidiaModel, self).__init__()

        # elu=Expenential Linear Unit, similar to leaky Relu
        # skipping 1st hiddel layer (nomralization layer), as we have normalized the data

        # 데이터는 200 * 66 사이즈 이미지

        # Convolution Layers
        # kernel : 이미지 학습을 진행할 때 구간별 특징 값을 추출할 범위의 크기
        # stride : kernel 을 몇 칸씩 이동시킬 것인가

        # 하나의 컨볼루션 연산을 진행 했을 때의 크기 (out_channels,(image_volume_size-kernel_size+2*padding)/(stride)+1)
        # padding 을 사용하지 않기 때문에 padding 은 초기값 0을 가지게 되어 +2p 를 더하지 않는다
        # 소수점은 내림
        # 기본 사이즈 [16,1,3] batch(바뀔 수 있음) in_channel(1차원 배열이므로 1), 배열 사이즈(3개 있음)
        self.layer1 = nn.Sequential(
        nn.Conv1d(in_channels=1,out_channels=4,kernel_size=3,stride=1,padding=2), #[16,4,5]
        nn.Conv1d(in_channels=4,out_channels=16,kernel_size=3,stride=1,padding=2), #[16,16,7]
        nn.Conv1d(in_channels=16,out_channels=32,kernel_size=2,stride=1,padding=1), #[16,32,8]
        nn.Conv1d(in_channels=32,out_channels=64,kernel_size=2,stride=1,padding=1), #[16,64,9]
        nn.ReLU(inplace=True),
        nn.MaxPool1d(kernel_size=3, stride=1,padding=1), #[16,64,9]
        nn.Conv1d(in_channels=64,out_channels=128,kernel_size=2,stride=1,padding=1), #[16,128,9]
        nn.Conv1d(in_channels=128,out_channels=128,kernel_size=1,stride=1), #[16,128,9]
        nn.ReLU(inplace=True),
        nn.Conv1d(in_channels=128,out_channels=128,kernel_size=1,stride=1), #[16,128,9]
        nn.MaxPool1d(kernel_size=3, stride=1), #[16,128,8]
        )

        # Fully Connected Layers
        self.layer2 = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(in_features=128 * 8, out_features=420),
            nn.ReLU(inplace=True),
            nn.Linear(in_features=420, out_features=200),
            nn.ReLU(inplace=True),
            nn.Linear(in_features=200, out_features=100),
            nn.ReLU(inplace=True),
            nn.Linear(in_features=100, out_features=50),
            nn.ReLU(inplace=True),
            nn.Linear(in_features=50, out_features=30),
            nn.ReLU(inplace=True),

        )

        # Output Layer
        self.layer3 = nn.Sequential(
        nn.Linear(in_features=30, out_features=10),
        nn.Linear(in_features=10, out_features=4), # 0 : 못맞춤, n (1<=n<=3) : n트
        )

    def forward(self, x):
        #print("before",x.size())
        x = self.layer1(x) # 합성곱 연산을 하는 레이어
        #print("layer1",x.size())
        x = x.view(x.shape[0], -1) 
        #print("view",x.size())
        x = self.layer2(x) # fc layer, 답을 얻기 위한 레이어
        #print("layer2",x.size())
        x = self.layer3(x) # fc layer, 답을 얻기 위한 레이어
        #print("layer3",x.size())
        return x

In [ ]:
# 학습 데이터 생성 -------------------------------------------------------------
# DATASET 이란 학습/테스트 에 필요한 데이터를 쉽게 다룰 수 있도록 하는 클래스
class CustomDataset(Dataset): # pytorch 에서 직접 DataSet 을 만들어 사용하기 위해 정의한 함수, 기본적으로 __init__, __len__, __getitem__ 이 있어야 함 #(x_exam_Image, y_exam_Angle)
    def __init__(self, x_table, y_table):
        self.x_table = x_table
        self.y_table = y_table

    def __len__(self):
        return len(self.x_table) # 테이블 개수 리턴

    def __getitem__(self, index):
        #delaytime	base	exponents
        idx_record=[self.x_table.loc[index]['delaytime'],self.x_table.loc[index]['base'],self.x_table.loc[index]['exponents']]
        
        return torch.tensor([idx_record]), torch.tensor([self.y_table.loc[index]['try']]) # idx 에 해당하는 이미지와 라벨 리턴

In [ ]:
learnRate=0.0005
batch_size=4

In [ ]:
def learnProc():

  model = NvidiaModel().to(device) #NvidiaModel 불러오고 device(gpu or cpu) 설정


  train_dataset = CustomDataset(X_train, y_train) # 학습을 위한 데이터 셋 제작
  train_loader = DataLoader(train_dataset, batch_size=batch_size) # 실질적 학습에 필요한 데이터 로더 제작

  valid_dataset = CustomDataset(X_test, y_test) # 검증에 위한 데이터 셋 제작
  valid_loader = DataLoader(valid_dataset, batch_size=batch_size)# 실질적 검즈에 필요한 데이터 로더 제작

  optimizer = optim.Adam(model.parameters(), lr=learnRate) # 가장 적절한 가중치(w, b)를 찾아 loss 를 최소화 하기 위해 파라미터 값을 최적화하는 알고리즘, lr(learning rate) 를 조정함
  criterion = nn.CrossEntropyLoss().to(device) # loss 알고리즘, MSELoss : 평균 제곱 오차 손실함수로 예측값고 실제 값의 차이를 제곱한 값들의 평균값을 loss 로 가지는 알고리즘이다.

  scheduler = lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

  # EPOCHS = epochN
  EPOCHS = 200 # 전체적인 학습을 반복할 횟수
  train_losses = [] # 한 epoch 에서 batch size 로 나눠진 데이터들에 대해 학습결과로 나왔던 train loss 를 저장하는 리스트
  valid_losses = [] # 한 epoch 에서 batch size 로 나눠진 데이터들에 대해 학습결과로 나왔던 val(검증) loss 를 저장하는 리스트
  avg_train_losses = [] # 각 epoch 에서 train_losses 에 저장한 loss 들의 평균들을 저장하는 리스트
  avg_valid_losses = [] # 각 epoch 에서 val_losses 에 저장한 loss 들의 평균들을 저장하는 리스트

  for epoch in range(1, EPOCHS + 1): # epoch 만큼 반복
      # 학습
      model.train() # NvidiaMODEL 을 학습시키기 위해 설정하는 코드

      for batch_idx, (data, target) in enumerate(train_loader): # dataloader 에서 각 batch size 크기의 데이터와 idx , 실제 각도 값을 불러옴
          data, target = data.to(device).float(), target.to(device).float() # cpu 또는 gpu 에서 사용할 수있도록 설정
          optimizer.zero_grad() #옵티마이저를 0으로 초기화, gradient(기울기) 가 0에 최대한 가까워야하는데 zero_grad() 를 하지 않을 경우 기울기가 이상하게 학습될 수 있음

          print(data,target)

          output = model(data) # 데이터를 모델에 넣고 학습을 했을 때의 결과
          
          
          loss = criterion(output, target)
          #if batch_idx==0:
            #print("output")
            #print(output)
            #print("target")
            #print(target)

          #loss = criterion(output.to(torch.float32), target.to(torch.float32)) # 손실함수(MSE)를 사용하여 예측 값과 실제 정답을 비교한 loss 계산

          loss.backward() # 역전파를 사용하여 가중치 수정, 역전파란 계산 결과와 오차를 가지고 미분을 사용하여 가중치 계산, 미분을 사용할 때 chain rule 을 통해 계산을 해서 구할 수 있음
          optimizer.step() # 역전파를 통해 업데이트 된 가중치에 따라 파라미터 조정, lr 값이 바뀜

          train_losses.append(loss.item()) # loss 값을 리스트에 추가

      scheduler.step()

      # 검증
      model.eval() # 학습한 모델을 테스트 데이터를 통해 검증을 돌리기 위해 설정하는 코드

      with torch.no_grad(): #가중치 업데이트를 그만하는 코드
        for data, target in valid_loader: # dataloader 에서 이미지, 라벨, idx 값을 불러옴 (train 때와 동일)
            data, target = data.to(device).float(), target.to(device).float() # cpu 또는 gpu 에서 사용할 수있도록 설정


            output = model(data) # 데이터를 모델에 넣고 검증을 했을 때의 결과
            #print("output")
            #print(output)
            #print("target")
            #print(target)

            loss = criterion(output, target) # 손실함수(MSE)를 사용하여 예측 값과 실제 정답을 비교한 loss 계산

            valid_losses.append(loss.item())# loss 값을 리스트에 추가

      train_loss = np.average(train_losses) # 학습할 때 나온loss 의 평균 값
      valid_loss = np.average(valid_losses) # 검증할 때 나온 loss 의 평균 값
      avg_train_losses.append(train_loss) # 학습 평균 loss 를 추가
      avg_valid_losses.append(valid_loss) # 검증 평균 loss 를 추가

      epoch_len = len(str(EPOCHS)) # epoch 의 글자수

      print_msg = (f'[{epoch:>{epoch_len}}/{EPOCHS:>{epoch_len}}] ' +
                      f'train_loss: {train_loss:.5f} ' +
                      f'valid_loss: {valid_loss:.5f}') # epoch 당 loss를 출력하기 위한 메세지 설정

      print(print_msg) #메세지 출력

      train_losses = [] # epoch 당 학습 loss 리스트 초기화
      valid_losses = [] # epoch 당 검증 loss 리스트 초기화

      # early stopping 을 사용하지 않음
      #if early_stopping.early_stop: # 조기 종료 여부가 true 라면
      #    print("Early stopping") #조기 종료 출력
      #    break #조기 종료

  return {'loss':avg_train_losses, 'val_loss':avg_valid_losses} # 학습 loss 와 검증 loss 리턴


In [ ]:
X_train.reset_index(inplace=True)
y_train.reset_index(inplace=True)

X_test.reset_index(inplace=True)
y_test.reset_index(inplace=True)

In [ ]:
learnProc()

!!
tensor([[[57.,  5.,  1.]],

        [[16.,  2.,  4.]],

        [[28.,  1.,  4.]],

        [[ 7.,  3.,  1.]],

        [[46.,  5.,  5.]],

        [[20.,  3.,  2.]],

        [[49.,  1.,  5.]]], device='cuda:0')
tensor([[1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.]], device='cuda:0')

In [ ]:


# 정확도는 좌우로 얼마나 퍼졌는지로 계산 0 에 가까울수록 정확도 높음?
# 실제 게임 데이터를 가지고 해야될듯


average_percent=0
for i in range(10):
    model = NvidiaModel().to(device) #NvidiaModel 불러오고 device(gpu or cpu) 설정
    model.eval()
    with torch.no_grad():
        input_x = torch.tensor([[[3,4,3]]]).to(device).float()
        output_y = model(input_x)
        print(f"i : {i}, output : {output_y}")
        average_percent+=output_y[0][0].item()
        


In [ ]:
average_percent

In [ ]:
average_percent/10